In [0]:
dbutils.widgets.text("entity", "ALL")

entity_param = dbutils.widgets.get("entity").strip()

print(f"Job parameter: {entity_param}")

In [0]:
import json

config_path = "/Volumes/odata-catlog/odata-metadata/odata-metadata/config.json"

config_text = dbutils.fs.head(config_path)
config = json.loads(config_text)

all_entities = config["entities"]

print(f"Entities available in config: {len(all_entities)}")

In [0]:
if entity_param.upper() == "ALL":

    entities_to_process = all_entities

else:

    matching_entities = [
        entity
        for entity in all_entities
        if entity.lower() == entity_param.lower()
    ]

    if not matching_entities:
        raise Exception(
            f"Entity '{entity_param}' was not found in config.json"
        )

    entities_to_process = matching_entities

print("Entities to process:")

for entity in entities_to_process:
    print(f" - {entity}")

In [0]:
landing_root = "/Volumes/odata-catlog/landing/landing-volume"

bronze_root = "/Volumes/odata-catlog/bronze/bronze-volume"

for entity in entities_to_process:

    print(f"\n{'='*60}")
    print(f"Processing: {entity}")
    print(f"{'='*60}")

    landing_path = f"{landing_root}/{entity}"

    print(f"Reading Landing data from:")
    print(landing_path)

    try:

        df = spark.read.json(landing_path)

    except Exception as e:

        print(f"Could not read Landing data for {entity}")
        print(str(e))
        continue

    record_count = df.count()

    print(f"Records read from Landing: {record_count}")

    if record_count == 0:

        print(f"No records found for {entity}")
        continue

    print("Schema:")
    df.printSchema()

    bronze_path = f"{bronze_root}/{entity}"

    print(f"Writing Bronze data to:")
    print(bronze_path)

    (
        df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .save(bronze_path)
    )

    print(f"Bronze write completed for {entity}")

In [0]:
Customers_path = f"{bronze_root}/Customers"

Customers_df = (
    spark.read
    .format("delta")
    .load(Customers_path)
)

print(f"Customers Bronze records: {Customers_df.count()}")

display(Customers_df)